## CELL 0: GLOBAL SEED + IMPORTS

In [ ]:
# ==============================================================================
# CELL 0: GLOBAL SEED + IMPORTS
# ==============================================================================
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')


def seed_everything(seed=42):
    print(f"[*] Locking random seeds to {seed}...")
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("[OK] Seeds locked.")


seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Device: {DEVICE}")


## CELL 1: HYBRID ARCHITECTURE  —  3 private encoders  +  1 shared cross-modal

In [ ]:
# ==============================================================================
# CELL 1: HYBRID ARCHITECTURE  —  3 private encoders  +  1 shared cross-modal
#
# CHANGES vs v3:
#   - PrivateEncoderECG: per-band LayerNorm BEFORE band-mix sum.  Stops one band
#     from dominating the softmax-weighted sum just because its activations are
#     larger in scale.
#   - LearnableWavelet1D exposes .lo and .hi so the loss can apply a QMF penalty.
#
# PATCHING: patch_len = 30  =>  num_patches = 3750 // 30 = 125
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F


# ── db6 wavelet decomposition coefficients ───────────────────────────────────
DB6_DEC_LO = torch.tensor([
    -0.00107730108499558,
     0.00477725751101065,
     0.000553842200993802,
    -0.031582039318031156,
     0.027522865530016288,
     0.09750160558707936,
    -0.12976686756709563,
    -0.22626469396516913,
     0.31525035170924843,
     0.7511339080215775,
     0.4946238903983854,
     0.11154074335008017,
], dtype=torch.float32)


def qmf(lo: torch.Tensor) -> torch.Tensor:
    """Quadrature-mirror high-pass from low-pass: hi[i] = (-1)^i * lo[N-1-i]."""
    n = lo.shape[-1]
    device = lo.device
    sign = torch.tensor([(-1.0) ** i for i in range(n)],
                        dtype=lo.dtype, device=device)
    return sign * lo.flip(-1)


def _conv_same(x: torch.Tensor, w: torch.Tensor, dilation: int = 1) -> torch.Tensor:
    """1D conv with same-length output via asymmetric padding."""
    f = w.shape[-1]
    pad_total = (f - 1) * dilation
    pad_left  = pad_total // 2
    pad_right = pad_total - pad_left
    x_pad = F.pad(x, (pad_left, pad_right))
    return F.conv1d(x_pad, w, dilation=dilation)


class LearnableWavelet1D(nn.Module):
    """
    Stationary (undecimated) wavelet decomposition with LEARNABLE filters.
    Initialised from db6.  Each level uses dilation 2^level (no decimation).

    Input  : x [B, L]
    Output : bands [B, K, L]   where K = n_levels + 1
             ordering: [D1, D2, ..., D_n_levels, A_n_levels]
    """
    def __init__(self, n_levels: int = 3, init_lo: torch.Tensor = DB6_DEC_LO):
        super().__init__()
        self.n_levels   = n_levels
        self.filter_len = init_lo.shape[-1]
        init_hi         = qmf(init_lo)

        self.lo = nn.ParameterList([
            nn.Parameter(init_lo.clone().view(1, 1, -1)) for _ in range(n_levels)
        ])
        self.hi = nn.ParameterList([
            nn.Parameter(init_hi.clone().view(1, 1, -1)) for _ in range(n_levels)
        ])

    @property
    def num_bands(self) -> int:
        return self.n_levels + 1

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)                                                  # [B,1,L]
        details = []
        current = x
        for lvl in range(self.n_levels):
            d = 2 ** lvl
            lo_out = _conv_same(current, self.lo[lvl], dilation=d)
            hi_out = _conv_same(current, self.hi[lvl], dilation=d)
            details.append(hi_out)
            current = lo_out
        bands = details + [current]
        return torch.cat(bands, dim=1)                                      # [B,K,L]

    def qmf_penalty(self) -> torch.Tensor:
        """
        Sum over levels of MSE between hi[k] and qmf(lo[k]).
        Keeps the learnable filters near a valid orthogonal wavelet basis.
        """
        loss = 0.0
        for k in range(self.n_levels):
            lo_k = self.lo[k].squeeze()                                     # [filter_len]
            hi_k = self.hi[k].squeeze()
            expected_hi = qmf(lo_k)
            loss = loss + F.mse_loss(hi_k, expected_hi)
        return loss / self.n_levels


# ── Private encoder for "raw" modalities (PPG, ABP) ──────────────────────────
class PrivateEncoderRaw(nn.Module):
    def __init__(self, seq_len=3750, patch_len=30, d_model=128,
                 n_heads=8, n_layers=2, dropout=0.1):
        super().__init__()
        self.patch_len   = patch_len
        self.d_model     = d_model
        self.num_patches = seq_len // patch_len

        self.patch_proj = nn.Linear(patch_len, d_model)
        self.pos_embed  = nn.Parameter(
            torch.randn(1, self.num_patches, d_model) * 0.02)
        self.dropout    = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm        = nn.LayerNorm(d_model)

    def forward(self, x):
        patches = x.unfold(dimension=-1, size=self.patch_len, step=self.patch_len)
        z = self.patch_proj(patches) + self.pos_embed
        z = self.dropout(z)
        z = self.transformer(z)
        z = self.norm(z)
        return z


# ── Private encoder for ECG (wavelet front-end + per-band LayerNorm) ─────────
class PrivateEncoderECG(nn.Module):
    """
      raw ECG -> learnable wavelet (K bands)
              -> patch each band -> per-band projection -> per-band LayerNorm
              -> softmax-weighted sum over bands
              -> 2-layer transformer -> z [B, P, d_model]
    """
    def __init__(self, seq_len=3750, patch_len=30, d_model=128,
                 n_heads=8, n_layers=2, dropout=0.1,
                 n_wavelet_levels=3):
        super().__init__()
        self.patch_len    = patch_len
        self.d_model      = d_model
        self.num_patches  = seq_len // patch_len

        self.wavelet = LearnableWavelet1D(n_levels=n_wavelet_levels)
        self.K       = self.wavelet.num_bands

        self.band_proj = nn.ModuleList([
            nn.Linear(patch_len, d_model) for _ in range(self.K)
        ])
        # NEW in v4:  per-band LayerNorm — stops one band dominating by scale
        self.band_norm = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(self.K)
        ])
        self.band_logits = nn.Parameter(torch.zeros(self.K))

        self.pos_embed = nn.Parameter(
            torch.randn(1, self.num_patches, d_model) * 0.02)
        self.dropout   = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm        = nn.LayerNorm(d_model)

    def forward(self, x):
        bands   = self.wavelet(x)                                          # [B,K,L]
        patches = bands.unfold(dimension=-1, size=self.patch_len,
                               step=self.patch_len)                        # [B,K,P,patch_len]

        proj = []
        for k in range(self.K):
            p_k = self.band_proj[k](patches[:, k])                         # [B,P,D]
            p_k = self.band_norm[k](p_k)                                   # NEW: per-band norm
            proj.append(p_k)
        proj = torch.stack(proj, dim=1)                                    # [B,K,P,D]

        w = F.softmax(self.band_logits, dim=0)                             # [K]
        z = (proj * w.view(1, -1, 1, 1)).sum(dim=1)                        # [B,P,D]

        z = z + self.pos_embed
        z = self.dropout(z)
        z = self.transformer(z)
        z = self.norm(z)
        return z


# ── Shared cross-modal encoder ───────────────────────────────────────────────
class SharedCrossModalEncoder(nn.Module):
    def __init__(self, d_model=128, n_heads=8, n_layers=3,
                 dropout=0.1, num_patches=125):
        super().__init__()
        self.d_model        = d_model
        self.num_patches    = num_patches
        self.modality_embed = nn.Embedding(3, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm        = nn.LayerNorm(d_model)

    def forward(self, z_ppg, z_abp, z_ecg):
        B, P, D = z_ppg.shape
        device  = z_ppg.device

        m_ppg = self.modality_embed(torch.zeros(1, dtype=torch.long, device=device)).view(1,1,D)
        m_abp = self.modality_embed(torch.ones(1,  dtype=torch.long, device=device)).view(1,1,D)
        m_ecg = self.modality_embed(torch.full((1,), 2, dtype=torch.long, device=device)).view(1,1,D)

        z = torch.cat([z_ppg + m_ppg, z_abp + m_abp, z_ecg + m_ecg], dim=1) # [B, 3P, D]
        z = self.transformer(z)
        z = self.norm(z)
        return z[:, 0:P], z[:, P:2*P], z[:, 2*P:3*P]


class HybridEncoderBundle(nn.Module):
    def __init__(self, enc_ppg, enc_abp, enc_ecg, shared):
        super().__init__()
        self.enc_ppg = enc_ppg
        self.enc_abp = enc_abp
        self.enc_ecg = enc_ecg
        self.shared  = shared

    def forward(self, x_ppg, x_abp, x_ecg):
        z_p = self.enc_ppg(x_ppg)
        z_a = self.enc_abp(x_abp)
        z_e = self.enc_ecg(x_ecg)
        return self.shared(z_p, z_a, z_e)


print("Cell 1 ready.")


## CELL 2: HYBRID PRETRAINER  (SimMTM v4)

In [ ]:
# ==============================================================================
# CELL 2: HYBRID PRETRAINER  (SimMTM v4)
#
# CHANGES vs v3:
#   - Adds a CLEAN-VIEW reconstruction path: each modality has a second
#     decoder head (`dec_clean_*`) that decodes the UNMASKED shared-block
#     output directly to the raw signal.  This gives the model a direct
#     signal-to-signal MSE objective and stops the masked-region collapse
#     we saw in v3.  Costs nothing extra at inference; it's just an extra
#     loss term during training.
#   - Forward returns both `x_hat_*` (SimMTM aggregate, as before) AND
#     `x_clean_*` (clean-path reconstruction).
#   - Exposes the ECG wavelet for the QMF loss.
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F


class HybridPretrainerV4(nn.Module):
    def __init__(self,
                 enc_ppg, enc_abp, enc_ecg, shared,
                 d_model=128, patch_len=30, num_patches=125,
                 M=3, tau=0.2,
                 mask_ratio=0.40, block_patches=6):                        # ← default lowered
        super().__init__()
        self.enc_ppg = enc_ppg
        self.enc_abp = enc_abp
        self.enc_ecg = enc_ecg
        self.shared  = shared

        self.d_model       = d_model
        self.patch_len     = patch_len
        self.num_patches   = num_patches
        self.M             = M
        self.tau           = tau
        self.mask_ratio    = mask_ratio
        self.block_patches = block_patches

        # Intra-modality pooling heads (on PRIVATE output)
        self.intra_pool_ppg = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.intra_pool_abp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.intra_pool_ecg = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))

        # Cross-modal projection heads (on SHARED output)
        self.proj_ppg = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.proj_abp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.proj_ecg = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))

        # SimMTM-aggregate decoders (as in v3)
        self.dec_ppg = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, patch_len))
        self.dec_abp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, patch_len))
        self.dec_ecg = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, patch_len))

        # NEW: CLEAN-view decoders.  Decode unmasked shared output -> raw signal.
        # Same architecture, separate weights — keeps the two objectives decoupled.
        self.dec_clean_ppg = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, patch_len))
        self.dec_clean_abp = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, patch_len))
        self.dec_clean_ecg = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, patch_len))

    # ── Block masking ────────────────────────────────────────────────────────
    def _block_mask(self, x):
        B, L      = x.shape
        block_len = self.patch_len * self.block_patches
        n_blocks  = max(1, int((self.mask_ratio * L) / block_len))
        masked    = x.clone()
        max_start = L - block_len
        for b in range(B):
            starts = torch.randint(0, max_start + 1, (n_blocks,), device=x.device)
            for s in starts:
                masked[b, s:s + block_len] = 0.0
        return masked

    # ── Reps ─────────────────────────────────────────────────────────────────
    def _intra_rep(self, z_priv, modality):
        pooled = z_priv.mean(dim=1)
        if   modality == 'ppg': return self.intra_pool_ppg(pooled)
        elif modality == 'abp': return self.intra_pool_abp(pooled)
        elif modality == 'ecg': return self.intra_pool_ecg(pooled)
        else: raise ValueError(modality)

    def _cross_rep(self, z_shared, modality):
        pooled = z_shared.mean(dim=1)
        if   modality == 'ppg': return self.proj_ppg(pooled)
        elif modality == 'abp': return self.proj_abp(pooled)
        elif modality == 'ecg': return self.proj_ecg(pooled)
        else: raise ValueError(modality)

    # ── Full encode (private x3 + shared) ────────────────────────────────────
    def _encode_full(self, x_ppg, x_abp, x_ecg):
        z_p = self.enc_ppg(x_ppg)
        z_a = self.enc_abp(x_abp)
        z_e = self.enc_ecg(x_ecg)
        z_p_s, z_a_s, z_e_s = self.shared(z_p, z_a, z_e)
        return (z_p, z_a, z_e), (z_p_s, z_a_s, z_e_s)

    # ── Token-level clean reconstruction (new in v4) ────────────────────────
    def _clean_decode(self, z_shared, decoder, B):
        """z_shared: [B, P, D]  ->  x_clean: [B, L]"""
        patches = decoder(z_shared)                                        # [B, P, patch_len]
        return patches.reshape(B, -1)

    # ── SimMTM aggregation per modality ──────────────────────────────────────
    def _simmtm_aggregate(self, z_orig_s, z_masked_s_list,
                          s_orig, s_masked_list, decoder, B):
        z_all   = torch.cat([z_orig_s] + z_masked_s_list, dim=0)
        s_all   = torch.cat([s_orig]   + s_masked_list,   dim=0)
        s_norm  = F.normalize(s_all, p=2, dim=-1)
        R       = torch.mm(s_norm, s_norm.T)
        D_total = (self.M + 1) * B

        x_hat_list = []
        for i in range(B):
            idx_others = torch.tensor(
                [j for j in range(D_total) if j != i], device=z_orig_s.device)
            sim_i   = R[i, idx_others]
            w_i     = F.softmax(sim_i / self.tau, dim=0)
            z_hat_i = (w_i.view(-1, 1, 1) * z_all[idx_others]).sum(0)
            x_hat_list.append(z_hat_i)

        z_hat        = torch.stack(x_hat_list, dim=0)
        pred_patches = decoder(z_hat)
        x_hat        = pred_patches.reshape(B, -1)
        return x_hat, s_all

    # ── Forward ──────────────────────────────────────────────────────────────
    def forward(self, x_ppg, x_abp, x_ecg):
        B = x_ppg.shape[0]

        # 1. CLEAN path
        (zp_o, za_o, ze_o), (zps_o, zas_o, zes_o) = self._encode_full(
            x_ppg, x_abp, x_ecg)

        s_ppg_o = self._intra_rep(zp_o, 'ppg')
        s_abp_o = self._intra_rep(za_o, 'abp')
        s_ecg_o = self._intra_rep(ze_o, 'ecg')

        c_ppg = self._cross_rep(zps_o, 'ppg')
        c_abp = self._cross_rep(zas_o, 'abp')
        c_ecg = self._cross_rep(zes_o, 'ecg')

        # NEW: clean-view direct reconstruction
        x_clean_ppg = self._clean_decode(zps_o, self.dec_clean_ppg, B)
        x_clean_abp = self._clean_decode(zas_o, self.dec_clean_abp, B)
        x_clean_ecg = self._clean_decode(zes_o, self.dec_clean_ecg, B)

        # 2. M masked views
        zps_masked, zas_masked, zes_masked = [], [], []
        s_ppg_masked, s_abp_masked, s_ecg_masked = [], [], []

        for _ in range(self.M):
            x_pm = self._block_mask(x_ppg)
            x_am = self._block_mask(x_abp)
            x_em = self._block_mask(x_ecg)
            (zp_m, za_m, ze_m), (zps_m, zas_m, zes_m) = self._encode_full(
                x_pm, x_am, x_em)
            zps_masked.append(zps_m)
            zas_masked.append(zas_m)
            zes_masked.append(zes_m)
            s_ppg_masked.append(self._intra_rep(zp_m, 'ppg'))
            s_abp_masked.append(self._intra_rep(za_m, 'abp'))
            s_ecg_masked.append(self._intra_rep(ze_m, 'ecg'))

        # 3. SimMTM aggregation
        x_hat_ppg, s_all_ppg = self._simmtm_aggregate(
            zps_o, zps_masked, s_ppg_o, s_ppg_masked, self.dec_ppg, B)
        x_hat_abp, s_all_abp = self._simmtm_aggregate(
            zas_o, zas_masked, s_abp_o, s_abp_masked, self.dec_abp, B)
        x_hat_ecg, s_all_ecg = self._simmtm_aggregate(
            zes_o, zes_masked, s_ecg_o, s_ecg_masked, self.dec_ecg, B)

        return dict(
            # SimMTM aggregate recon (kept from v3)
            x_hat_ppg=x_hat_ppg, x_hat_abp=x_hat_abp, x_hat_ecg=x_hat_ecg,
            # NEW: clean-path recon
            x_clean_ppg=x_clean_ppg, x_clean_abp=x_clean_abp, x_clean_ecg=x_clean_ecg,
            # Contrastive bookkeeping
            s_all_ppg=s_all_ppg, s_all_abp=s_all_abp, s_all_ecg=s_all_ecg,
            c_ppg=c_ppg, c_abp=c_abp, c_ecg=c_ecg,
            B=B,
        )


print("Cell 2 ready.")


## CELL 3: HYBRID LOSS V6

In [ ]:
# ==============================================================================
# CELL 3: HYBRID LOSS V6  (PPG-morphology-focused)
#
# CHANGES vs v5:
#   1. CP-PPG fiducial loss on PPG     (sys peak + dicrotic notch + dia peak)
#   2. Pearson correlation loss        (per-modality, replaces some MSE weight)
#   3. Soft-DTW on PPG                 (morphology curvature)
#   4. Multi-resolution STFT loss      (REPLACES the old single-scale FFT loss)
#   5. (mask_ratio is changed in Cell 2, not here)
#
# Loss equation:
#   L_total =
#       Σ_m  [ 0.8 * L_mse(m)
#            + 0.2 * (1 - PCC(m))
#            + lam_mrstft * L_mr_stft(m)
#            + lam_peak   * L_peak(m)
#            + lam_clean  * L_clean(m) ]
#     + Σ_{m in {ppg, abp}}  [ lam_d1 * L_d1(m) + lam_d2 * L_d2(m) ]
#     + lam_fid     * L_fiducial(ppg)                    ← NEW
#     + lam_sdtw    * L_softdtw(ppg)                     ← NEW
#     + lam_intra   * L_simmtm_intra
#     + lam_cross   * L_cross_modal
#     + lam_wavelet * L_wavelet_qmf
#
# NOTES:
#   - Fiducial indices are detected on the GROUND-TRUTH PPG only, then the
#     predicted signal is sampled at those indices.  This is what CP-PPG does
#     and avoids unstable peak-detection on noisy early reconstructions.
#   - Soft-DTW requires `pip install pytorch-softdtw-cuda`.  If unavailable,
#     set lam_sdtw=0.0 and the term is skipped.
#   - The single log-FFT term from v5 is REMOVED; multi-res STFT subsumes it.
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from sdtw_cuda_loss import SoftDTW
    _HAS_SOFTDTW = True
except ImportError:
    _HAS_SOFTDTW = False
    print("[!] pytorch-softdtw-cuda not installed; lam_sdtw will be ignored.")


# ── Fiducial point detection on raw PPG ──────────────────────────────────────
def _detect_ppg_fiducials(x, fs=125, min_hr_bpm=40, max_hr_bpm=180):
    """
    Detect systolic peak, dicrotic notch, diastolic peak indices per beat.

    Algorithm:
      systolic peak  : local max of x  with min spacing = 60*fs/max_hr_bpm
      dicrotic notch : local max of -x'' (second derivative) in the window
                       [sys_i, sys_i + 0.5*(sys_{i+1}-sys_i)]
      diastolic peak : local max of x in the window
                       [notch_i, sys_{i+1}]

    Returns: (idx_sys, idx_notch, idx_dia) each as a 1-D LongTensor on x.device
             — indices into x  (shape [L]).  May be empty if detection fails.

    For batch use, call once per sample in a loop.
    """
    import numpy as np
    from scipy.signal import find_peaks

    x_np = x.detach().cpu().numpy()
    L    = len(x_np)
    min_dist = int(60 * fs / max_hr_bpm)         # samples

    # Systolic peaks
    sys_idx, _ = find_peaks(x_np, distance=min_dist)
    if len(sys_idx) < 2:
        empty = torch.empty(0, dtype=torch.long, device=x.device)
        return empty, empty, empty

    # Second derivative for notch
    d2 = np.diff(x_np, n=2, prepend=x_np[0], append=x_np[-1])

    notch_idx = []
    dia_idx   = []
    for i in range(len(sys_idx) - 1):
        s_start, s_end = sys_idx[i], sys_idx[i + 1]
        mid = s_start + (s_end - s_start) // 2

        # Notch: max of -d2 in [s_start, mid]
        win = -d2[s_start:mid]
        if len(win) > 2:
            notch_local, _ = find_peaks(win)
            if len(notch_local) > 0:
                notch_pos = s_start + int(notch_local[0])
                notch_idx.append(notch_pos)

                # Diastolic peak: max of x in [notch_pos, s_end]
                dia_win = x_np[notch_pos:s_end]
                if len(dia_win) > 2:
                    dia_local, _ = find_peaks(dia_win)
                    if len(dia_local) > 0:
                        dia_idx.append(notch_pos + int(dia_local[0]))

    return (torch.tensor(sys_idx,   dtype=torch.long, device=x.device),
            torch.tensor(notch_idx, dtype=torch.long, device=x.device),
            torch.tensor(dia_idx,   dtype=torch.long, device=x.device))


class HybridLossV6(nn.Module):
    def __init__(self,
                 tau=0.2, M=3,
                 # MSE / PCC balance (per modality)
                 lam_mse=0.8, lam_pcc=0.2,
                 # Other reconstruction terms
                 lam_mrstft=0.5, lam_peak=0.3, lam_clean=1.0,
                 # PPG morphology terms
                 lam_d1=0.3, lam_d2=0.5,
                 lam_fid=0.05,                                # NEW — CP-PPG
                 lam_sdtw=0.01,                               # NEW — soft-DTW
                 # Notch/diastolic re-weighting inside L_fid
                 fid_w_sys=1.0, fid_w_notch=2.0, fid_w_dia=1.0,
                 # Contrastive
                 lam_intra=1.0, lam_cross=0.1,
                 # Wavelet
                 lam_wavelet=0.01,
                 # Misc
                 peak_quantile=0.85, fs=125,
                 stft_ffts=(128, 256, 512, 1024)):            # NEW — multi-res
        super().__init__()
        self.tau          = tau
        self.M            = M
        self.lam_mse      = lam_mse
        self.lam_pcc      = lam_pcc
        self.lam_mrstft   = lam_mrstft
        self.lam_peak     = lam_peak
        self.lam_clean    = lam_clean
        self.lam_d1       = lam_d1
        self.lam_d2       = lam_d2
        self.lam_fid      = lam_fid
        self.lam_sdtw     = lam_sdtw
        self.fid_w_sys    = fid_w_sys
        self.fid_w_notch  = fid_w_notch
        self.fid_w_dia    = fid_w_dia
        self.lam_intra    = lam_intra
        self.lam_cross    = lam_cross
        self.lam_wavelet  = lam_wavelet
        self.peak_quantile= peak_quantile
        self.fs           = fs
        self.stft_ffts    = stft_ffts

        if _HAS_SOFTDTW and lam_sdtw > 0:
            self.softdtw = SoftDTW(use_cuda=torch.cuda.is_available(),
                                   gamma=0.1, normalize=False)
        else:
            self.softdtw = None

    # ── Multi-resolution STFT loss ───────────────────────────────────────────
    def _mr_stft_loss(self, x, x_hat):
        """
        Sum over FFT sizes of  [ log-mag L1  +  spectral convergence ].
        Each scale uses hop = n_fft / 4.
        """
        loss = 0.0
        for n_fft in self.stft_ffts:
            hop = n_fft // 4
            window = torch.hann_window(n_fft, device=x.device)
            X     = torch.stft(x,     n_fft=n_fft, hop_length=hop,
                               window=window, return_complex=True)
            X_hat = torch.stft(x_hat, n_fft=n_fft, hop_length=hop,
                               window=window, return_complex=True)
            mag, mag_hat = X.abs(), X_hat.abs()

            log_l1 = F.l1_loss(torch.log1p(mag_hat), torch.log1p(mag))
            spec_conv = (mag - mag_hat).norm(p='fro') / (mag.norm(p='fro') + 1e-8)
            loss = loss + log_l1 + spec_conv
        return loss / len(self.stft_ffts)

    # ── Pearson correlation loss (batch-averaged) ────────────────────────────
    @staticmethod
    def _pcc_loss(x, x_hat):
        """1 - PCC, computed per-sample then averaged."""
        xm     = x     - x.mean(dim=-1, keepdim=True)
        xm_hat = x_hat - x_hat.mean(dim=-1, keepdim=True)
        num = (xm * xm_hat).sum(dim=-1)
        den = torch.sqrt((xm ** 2).sum(dim=-1) *
                         (xm_hat ** 2).sum(dim=-1)) + 1e-8
        pcc = num / den
        return (1.0 - pcc).mean()

    # ── Peak-region MSE (scale-invariant, unchanged) ────────────────────────
    @staticmethod
    def _peak_loss(x, x_hat, top_quantile=0.85):
        abs_x = x.abs()
        thr   = torch.quantile(abs_x, top_quantile, dim=-1, keepdim=True)
        mask  = (abs_x >= thr).float()
        sqerr = (x - x_hat) ** 2
        peak_mse = (sqerr * mask).sum(dim=-1) / mask.sum(dim=-1).clamp(min=1.0)
        scale    = x.abs().mean(dim=-1).clamp(min=1e-6)
        return (peak_mse / (scale ** 2)).mean()

    # ── Derivative losses (unchanged) ────────────────────────────────────────
    @staticmethod
    def _d1_loss(x, x_hat):
        d1_x     = x[..., 1:] - x[..., :-1]
        d1_x_hat = x_hat[..., 1:] - x_hat[..., :-1]
        return F.mse_loss(d1_x_hat, d1_x)

    @staticmethod
    def _d2_loss(x, x_hat):
        d2_x     = x[..., 2:]     - 2 * x[..., 1:-1]     + x[..., :-2]
        d2_x_hat = x_hat[..., 2:] - 2 * x_hat[..., 1:-1] + x_hat[..., :-2]
        return F.mse_loss(d2_x_hat, d2_x)

    # ── CP-PPG fiducial loss ─────────────────────────────────────────────────
    def _fiducial_loss(self, x_ppg, x_hat_ppg):
        """
        L_fid = mean over batch of weighted L1 amplitude errors at
                {systolic peak, dicrotic notch, diastolic peak} indices
                (detected on ground-truth PPG).

        Skips samples where fewer than 2 systolic peaks are detected.
        """
        B = x_ppg.shape[0]
        loss_terms = []

        for b in range(B):
            idx_sys, idx_notch, idx_dia = _detect_ppg_fiducials(
                x_ppg[b], fs=self.fs)

            sample_loss = 0.0
            n_terms     = 0

            if len(idx_sys) > 0:
                err = (x_hat_ppg[b, idx_sys] - x_ppg[b, idx_sys]).abs().mean()
                sample_loss = sample_loss + self.fid_w_sys * err
                n_terms += 1
            if len(idx_notch) > 0:
                err = (x_hat_ppg[b, idx_notch] - x_ppg[b, idx_notch]).abs().mean()
                sample_loss = sample_loss + self.fid_w_notch * err
                n_terms += 1
            if len(idx_dia) > 0:
                err = (x_hat_ppg[b, idx_dia] - x_ppg[b, idx_dia]).abs().mean()
                sample_loss = sample_loss + self.fid_w_dia * err
                n_terms += 1

            if n_terms > 0:
                loss_terms.append(sample_loss / n_terms)

        if len(loss_terms) == 0:
            return torch.tensor(0.0, device=x_ppg.device)
        return torch.stack(loss_terms).mean()

    # ── Soft-DTW on PPG ──────────────────────────────────────────────────────
    def _sdtw_loss(self, x_ppg, x_hat_ppg):
        """Soft-DTW requires [B, L, 1] inputs."""
        if self.softdtw is None:
            return torch.tensor(0.0, device=x_ppg.device)
        a = x_ppg.unsqueeze(-1)
        b = x_hat_ppg.unsqueeze(-1)
        return self.softdtw(a, b).mean() / x_ppg.shape[-1]   # length-normalised

    # ── Contrastive terms (unchanged) ────────────────────────────────────────
    def _intra_contrastive(self, s_all, B):
        s_norm = F.normalize(s_all, p=2, dim=-1)
        R      = torch.mm(s_norm, s_norm.T) / self.tau
        l = 0.0
        for i in range(B):
            pos_idx  = [i + m * B for m in range(1, self.M + 1)]
            neg_mask = torch.ones(R.shape[0], dtype=torch.bool, device=R.device)
            neg_mask[i] = False
            log_sum_exp = torch.logsumexp(R[i][neg_mask], dim=0)
            for p in pos_idx:
                l += -(R[i, p] - log_sum_exp)
        return l / (B * self.M)

    def _cross_modal_contrastive(self, c_ppg, c_abp, c_ecg):
        def info_nce(a, b):
            a_n = F.normalize(a, dim=-1)
            b_n = F.normalize(b, dim=-1)
            sim = a_n @ b_n.T / self.tau
            labels = torch.arange(sim.shape[0], device=sim.device)
            return 0.5 * (F.cross_entropy(sim, labels) +
                          F.cross_entropy(sim.T, labels))
        return (info_nce(c_ppg, c_abp) +
                info_nce(c_ppg, c_ecg) +
                info_nce(c_abp, c_ecg)) / 3.0

    # ── Forward ──────────────────────────────────────────────────────────────
    def forward(self, x_ppg, x_abp, x_ecg, out, ecg_wavelet=None):
        B = out['B']

        # ── Per-modality reconstruction terms ────────────────────────────────
        rec_terms = {}
        for m, (x, x_hat, x_clean) in [
            ('ppg', (x_ppg, out['x_hat_ppg'], out['x_clean_ppg'])),
            ('abp', (x_abp, out['x_hat_abp'], out['x_clean_abp'])),
            ('ecg', (x_ecg, out['x_hat_ecg'], out['x_clean_ecg'])),
        ]:
            l_mse    = F.mse_loss(x_hat, x)
            l_pcc    = self._pcc_loss(x, x_hat)
            l_mrstft = self._mr_stft_loss(x, x_hat)
            l_peak   = self._peak_loss(x, x_hat, top_quantile=self.peak_quantile)
            l_clean  = F.mse_loss(x_clean, x)
            rec_terms[m] = (l_mse, l_pcc, l_mrstft, l_peak, l_clean)

        l_mse_tot    = sum(t[0] for t in rec_terms.values())
        l_pcc_tot    = sum(t[1] for t in rec_terms.values())
        l_mrstft_tot = sum(t[2] for t in rec_terms.values())
        l_peak_tot   = sum(t[3] for t in rec_terms.values())
        l_clean_tot  = sum(t[4] for t in rec_terms.values())

        # ── Derivative losses (PPG + ABP only) ───────────────────────────────
        l_d1_ppg = self._d1_loss(x_ppg, out['x_clean_ppg'])
        l_d1_abp = self._d1_loss(x_abp, out['x_clean_abp'])
        l_d2_ppg = self._d2_loss(x_ppg, out['x_clean_ppg'])
        l_d2_abp = self._d2_loss(x_abp, out['x_clean_abp'])
        l_d1 = l_d1_ppg + l_d1_abp
        l_d2 = l_d2_ppg + l_d2_abp

        # ── PPG-specific morphology losses (NEW) ─────────────────────────────
        l_fid  = self._fiducial_loss(x_ppg, out['x_clean_ppg'])
        l_sdtw = self._sdtw_loss(x_ppg, out['x_clean_ppg'])

        # ── Contrastive ──────────────────────────────────────────────────────
        l_intra = (self._intra_contrastive(out['s_all_ppg'], B) +
                   self._intra_contrastive(out['s_all_abp'], B) +
                   self._intra_contrastive(out['s_all_ecg'], B))
        l_cross = self._cross_modal_contrastive(
            out['c_ppg'], out['c_abp'], out['c_ecg'])

        # ── Wavelet QMF (ECG only) ───────────────────────────────────────────
        if ecg_wavelet is not None:
            l_wavelet = ecg_wavelet.qmf_penalty()
        else:
            l_wavelet = torch.tensor(0.0, device=x_ppg.device)

        # ── Total ────────────────────────────────────────────────────────────
        total = (self.lam_mse    * l_mse_tot
               + self.lam_pcc    * l_pcc_tot
               + self.lam_mrstft * l_mrstft_tot
               + self.lam_peak   * l_peak_tot
               + self.lam_clean  * l_clean_tot
               + self.lam_d1     * l_d1
               + self.lam_d2     * l_d2
               + self.lam_fid    * l_fid                                 # NEW
               + self.lam_sdtw   * l_sdtw                                # NEW
               + self.lam_intra  * l_intra
               + self.lam_cross  * l_cross
               + self.lam_wavelet * l_wavelet)

        return total, dict(
            mse       = l_mse_tot.item(),
            pcc       = l_pcc_tot.item(),
            mrstft    = l_mrstft_tot.item(),
            peak      = l_peak_tot.item(),
            clean     = l_clean_tot.item(),
            d1        = l_d1.item(),
            d2        = l_d2.item(),
            fid       = l_fid.item(),                                    # NEW
            sdtw      = l_sdtw.item() if torch.is_tensor(l_sdtw) else 0.0,
            intra     = l_intra.item(),
            cross     = l_cross.item(),
            wavelet   = float(l_wavelet.item()) if torch.is_tensor(l_wavelet) else 0.0,
            ppg_mse   = rec_terms['ppg'][0].item(),
            abp_mse   = rec_terms['abp'][0].item(),
            ecg_mse   = rec_terms['ecg'][0].item(),
            ppg_pcc   = rec_terms['ppg'][1].item(),
            abp_pcc   = rec_terms['abp'][1].item(),
            ecg_pcc   = rec_terms['ecg'][1].item(),
        )


print("Cell 3 ready.")



## CELL 4: COMPILE MIMIC DATA INTO MEMMAP

In [ ]:
# ==============================================================================
# CELL 4: COMPILE MIMIC DATA INTO MEMMAP (RUN ONCE)
# ==============================================================================
import os
import numpy as np
from tqdm import tqdm


def compile_mimic_to_memmap(data_root_dir, output_dir, seq_len=3750):
    os.makedirs(output_dir, exist_ok=True)
    skip_flag = os.path.join(output_dir, 'n_samples.npy')
    if os.path.exists(skip_flag):
        n = int(np.load(skip_flag)[0])
        print(f"[*] MIMIC memmap already exists ({n} samples). Skipping.")
        return

    ecg_dir = os.path.join(data_root_dir, 'ecg')
    ppg_dir = os.path.join(data_root_dir, 'ppg')
    abp_dir = os.path.join(data_root_dir, 'abp')

    all_ecg = [f for f in os.listdir(ecg_dir) if f.endswith('.npy')]
    valid   = []
    for ecg_f in all_ecg:
        base  = ecg_f.replace('_ecg.npy', '').replace('.npy', '')
        ppg_f = f"{base}_ppg.npy"
        abp_f = f"{base}_abp.npy"
        if (os.path.exists(os.path.join(ppg_dir, ppg_f)) and
                os.path.exists(os.path.join(abp_dir, abp_f))):
            valid.append((ecg_f, ppg_f, abp_f))

    # ── First pass: count total segments across all patients ─────────────────
    print(f"[*] Counting total segments across {len(valid)} patients...")
    total_segments = 0
    per_patient_counts = []
    for ecg_f, _, _ in tqdm(valid):
        arr = np.load(os.path.join(ecg_dir, ecg_f))
        if arr.ndim == 1:
            n_seg = len(arr) // seq_len
        elif arr.ndim == 2:
            n_seg = arr.shape[0] if arr.shape[1] == seq_len else 0
        else:
            n_seg = 0
        per_patient_counts.append(n_seg)
        total_segments += n_seg

    N = total_segments
    print(f"[*] Total segments: {N}  (avg {N/len(valid):.1f} per patient)")

    # ── Allocate memmaps for ALL segments ────────────────────────────────────
    ecg_mm = np.memmap(os.path.join(output_dir, 'ecg.dat'),
                       dtype='float32', mode='w+', shape=(N, seq_len))
    ppg_mm = np.memmap(os.path.join(output_dir, 'ppg.dat'),
                       dtype='float32', mode='w+', shape=(N, seq_len))
    abp_mm = np.memmap(os.path.join(output_dir, 'abp.dat'),
                       dtype='float32', mode='w+', shape=(N, seq_len))

    def load_all_segments(path, expected_n):
        """Returns array of shape (expected_n, seq_len) with NaN cleaning."""
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 1:
            # Old-format: 1D array, reshape into segments
            arr = arr[:expected_n * seq_len].reshape(expected_n, seq_len)
        elif arr.ndim == 2:
            arr = arr[:expected_n, :seq_len]
        # NaN cleanup per segment
        for i in range(arr.shape[0]):
            seg = arr[i]
            m = np.nanmean(seg) if not np.all(np.isnan(seg)) else 0.0
            arr[i] = np.nan_to_num(seg, nan=m)
        return arr

    # ── Second pass: fill memmaps ────────────────────────────────────────────
    write_idx = 0
    for (ecg_f, ppg_f, abp_f), n_seg in tqdm(
            zip(valid, per_patient_counts), total=len(valid)):
        if n_seg == 0:
            continue
        ecg_segs = load_all_segments(os.path.join(ecg_dir, ecg_f), n_seg)
        ppg_segs = load_all_segments(os.path.join(ppg_dir, ppg_f), n_seg)
        abp_segs = load_all_segments(os.path.join(abp_dir, abp_f), n_seg)

        ecg_mm[write_idx:write_idx + n_seg] = ecg_segs
        ppg_mm[write_idx:write_idx + n_seg] = ppg_segs
        abp_mm[write_idx:write_idx + n_seg] = abp_segs
        write_idx += n_seg

    del ecg_mm, ppg_mm, abp_mm
    np.save(skip_flag, np.array([N]))
    print(f"[OK] MIMIC memmap saved to {output_dir}  ({N} segments)")

MIMIC_DATA_DIR = "/kaggle/input/datasets/...../mimic-bp/dataverse_files"

MIMIC_COMPILED = "/kaggle/working/mimic_memmap"

compile_mimic_to_memmap(MIMIC_DATA_DIR, MIMIC_COMPILED)


## CELL 5: TRIPLET PRETRAIN DATASET

In [ ]:
# ==============================================================================
# CELL 5: TRIPLET PRETRAIN DATASET — robust z-score per-segment
# ==============================================================================
import os
import numpy as np
import torch
from torch.utils.data import Dataset


class TripletPretrainDataset(Dataset):
    def __init__(self, compiled_dir, seq_len=3750):
        self.seq_len = seq_len
        N = int(np.load(os.path.join(compiled_dir, 'n_samples.npy'))[0])
        self.N = N

        self.data = {}
        for ch in ['ppg', 'abp', 'ecg']:
            self.data[ch] = np.memmap(
                os.path.join(compiled_dir, f'{ch}.dat'),
                dtype='float32', mode='r', shape=(N, seq_len))

        print(f"[*] TripletPretrainDataset: {N} aligned (PPG,ABP,ECG) triplets")

    def __len__(self):
        return self.N

    @staticmethod
    def _robust_z(x, clip=5.0):
        med = np.median(x)
        q75, q25 = np.percentile(x, [75, 25])
        iqr = (q75 - q25) + 1e-6
        z = (x - med) / iqr
        z = np.clip(z, -clip, clip)
        return z.astype(np.float32)

    def __getitem__(self, idx):
        ppg = self._robust_z(np.array(self.data['ppg'][idx]).copy())
        abp = self._robust_z(np.array(self.data['abp'][idx]).copy())
        ecg = self._robust_z(np.array(self.data['ecg'][idx]).copy())
        return (torch.from_numpy(ppg),
                torch.from_numpy(abp),
                torch.from_numpy(ecg))


print("Cell 5 ready.")


## CELL 6: PRETRAIN HYBRID V4

In [ ]:
# ==============================================================================
# CELL 6: PRETRAIN HYBRID V4
#
# Changes vs v3 (recap):
#   - mask_ratio default lowered  0.40 -> 0.25
#   - clean-recon loss term added (lam_clean=1.0)
#   - wavelet QMF loss term added (lam_wavelet=0.01) — ECG only
#   - per-band LayerNorm now inside PrivateEncoderECG (cell 1)
# ==============================================================================
import os
import time
import torch
from torch.utils.data import DataLoader, random_split


def train_hybrid_v4(compiled_dirs,                  # ← was `compiled_dir`, now a LIST
                    epochs=100, batch_size=32, lr=1e-4,
                    patience=20, M=3, tau=0.2,
                    mask_ratio=0.40, block_patches=6,
                    lam_freq=0.5, lam_peak=0.3,
                    lam_intra=1.0, lam_cross=0.1,
                    lam_clean=1.0,
                    lam_wavelet=0.01,
                    lam_var=1.0,                    # NEW
                    lam_cov=0.04,                   # NEW
                    lam_d1=0.3,                     # NEW
                    lam_d2=0.5,                     # NEW
                    peak_quantile=0.85,
                    d_model=128, n_heads=8,
                    private_layers=2, shared_layers=3,
                    n_wavelet_levels=3):
    DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SAVE_ENC  = "/kaggle/working/hybrid_v4_encoders.pth"
    SAVE_FULL = "/kaggle/working/hybrid_v4_full.pth"

    print(f"\n{'=' * 70}")
    print(f"  HYBRID v4 PRE-TRAINING  —  3 private + shared cross-modal")
    print(f"{'=' * 70}")
    print(f"[*] Device              : {DEVICE}")
    print(f"[*] M={M}  tau={tau}  mask_ratio={mask_ratio}  block_patches={block_patches}")
    print(f"[*] lam_freq={lam_freq}  lam_peak={lam_peak}  lam_clean={lam_clean}  "
          f"lam_wavelet={lam_wavelet}  lam_intra={lam_intra}  lam_cross={lam_cross}  "
          f"peak_q={peak_quantile}")
    print(f"[*] Architecture        : private={private_layers}L  shared={shared_layers}L  "
          f"wavelet_levels={n_wavelet_levels} (ECG only)  d_model={d_model}")

    # ── Dataset ──────────────────────────────────────────────────────────────
    from torch.utils.data import ConcatDataset
    sub_datasets = [TripletPretrainDataset(compiled_dir=d) for d in compiled_dirs]
    full_ds      = ConcatDataset(sub_datasets) if len(sub_datasets) > 1 else sub_datasets[0]
    print(f"[*] Combined {len(sub_datasets)} dataset(s) into {len(full_ds):,} triplets")
    val_size = int(0.1 * len(full_ds))
    train_ds, val_ds = random_split(
        full_ds, [len(full_ds) - val_size, val_size],
        generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              drop_last=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)
    print(f"[*] Train: {len(train_ds):,}  |  Val: {len(val_ds):,} triplets")

    # ── Build encoders ───────────────────────────────────────────────────────
    enc_ppg = PrivateEncoderRaw(
        seq_len=3750, patch_len=30, d_model=d_model,
        n_heads=n_heads, n_layers=private_layers).to(DEVICE)
    enc_abp = PrivateEncoderRaw(
        seq_len=3750, patch_len=30, d_model=d_model,
        n_heads=n_heads, n_layers=private_layers).to(DEVICE)
    enc_ecg = PrivateEncoderRaw(
        seq_len=3750, patch_len=30, d_model=d_model,
        n_heads=n_heads, n_layers=private_layers).to(DEVICE)
    shared  = SharedCrossModalEncoder(
        d_model=d_model, n_heads=n_heads, n_layers=shared_layers,
        num_patches=3750 // 30).to(DEVICE)

    bundle = HybridEncoderBundle(enc_ppg, enc_abp, enc_ecg, shared).to(DEVICE)
    model  = HybridPretrainerV4(
        enc_ppg=enc_ppg, enc_abp=enc_abp, enc_ecg=enc_ecg, shared=shared,
        d_model=d_model, patch_len=30, num_patches=3750 // 30,
        M=M, tau=tau, mask_ratio=mask_ratio, block_patches=block_patches,
    ).to(DEVICE)

    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[*] Trainable parameters: {total:,}")
    print(f"    - enc_ppg : {sum(p.numel() for p in enc_ppg.parameters()):,}")
    print(f"    - enc_abp : {sum(p.numel() for p in enc_abp.parameters()):,}")
    print(f"    - enc_ecg : {sum(p.numel() for p in enc_ecg.parameters()):,}  "
          f"(wavelet front-end + band LayerNorms)")
    print(f"    - shared  : {sum(p.numel() for p in shared.parameters()):,}")

    # ── Optimizer + scheduler ────────────────────────────────────────────────
    criterion = HybridLossV6(
    tau=tau, M=M,
    lam_mse=0.8, lam_pcc=0.2,           # NEW MSE/PCC split
    lam_mrstft=lam_freq,                 # renamed from lam_freq
    lam_peak=lam_peak,
    lam_clean=lam_clean,
    lam_d1=lam_d1, lam_d2=lam_d2,
    lam_fid=0.05,                        # NEW
    lam_sdtw=0.01,                       # NEW
    lam_intra=lam_intra, lam_cross=lam_cross,
    lam_wavelet=0.0,
    lam_var=lam_var, lam_cov=lam_cov,    # NEW
    peak_quantile=peak_quantile,
    fs=125,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    def lr_lambda(epoch):
        warmup = 5
        if epoch < warmup:
            return (epoch + 1) / warmup
        progress = (epoch - warmup) / max(1, epochs - warmup)
        return 0.5 * (1.0 + torch.cos(torch.tensor(progress * 3.14159)).item())

    scheduler  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    best_val   = float('inf')
    no_improve = 0
    history    = []

    LOSS_KEYS = ['mse', 'pcc', 'mrstft', 'peak', 'clean',
             'd1', 'd2', 'fid', 'sdtw',
             'intra', 'cross', 'wavelet',
             'ppg_mse', 'abp_mse', 'ecg_mse',
             'ppg_pcc', 'abp_pcc', 'ecg_pcc']

    for epoch in range(epochs):
        t0 = time.time()

        # ── Train ────────────────────────────────────────────────────────────
        model.train()
        agg_tr = {k: 0.0 for k in ['total'] + LOSS_KEYS}
        for x_ppg, x_abp, x_ecg in train_loader:
            x_ppg = x_ppg.to(DEVICE, non_blocking=True)
            x_abp = x_abp.to(DEVICE, non_blocking=True)
            x_ecg = x_ecg.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            out = model(x_ppg, x_abp, x_ecg)
            loss, parts = criterion(x_ppg, x_abp, x_ecg, out,
                                    ecg_wavelet=enc_ecg.wavelet)             # ← QMF loss source
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            agg_tr['total'] += loss.item()
            for k in LOSS_KEYS:
                agg_tr[k] += parts[k]

        scheduler.step()
        n_tr = len(train_loader)
        for k in agg_tr:
            agg_tr[k] /= n_tr

        # ── Validate ─────────────────────────────────────────────────────────
        model.eval()
        agg_vl = {k: 0.0 for k in ['total'] + LOSS_KEYS}
        with torch.no_grad():
            for x_ppg, x_abp, x_ecg in val_loader:
                x_ppg = x_ppg.to(DEVICE, non_blocking=True)
                x_abp = x_abp.to(DEVICE, non_blocking=True)
                x_ecg = x_ecg.to(DEVICE, non_blocking=True)
                out = model(x_ppg, x_abp, x_ecg)
                loss, parts = criterion(x_ppg, x_abp, x_ecg, out,
                                        ecg_wavelet=enc_ecg.wavelet)
                agg_vl['total'] += loss.item()
                for k in LOSS_KEYS:
                    agg_vl[k] += parts[k]
        n_v = len(val_loader)
        for k in agg_vl:
            agg_vl[k] /= n_v

        elapsed = (time.time() - t0) / 60
        lr_now  = optimizer.param_groups[0]['lr']

        print(f"  Ep [{epoch+1:03d}/{epochs}]  "f"val={agg_vl['total']:.4f}  "f"(mse={agg_vl['mse']:.4f} pcc={agg_vl['pcc']:.4f} "f"mrstft={agg_vl['mrstft']:.4f} fid={agg_vl['fid']:.4f} "f"sdtw={agg_vl['sdtw']:.4f} d2={agg_vl['d2']:.4f})  "f"ppg[mse={agg_vl['ppg_mse']:.4f} pcc={agg_vl['ppg_pcc']:.4f}]  "f"lr={lr_now:.2e}  t={elapsed:.1f}m")

        
        history.append(dict(epoch=epoch + 1, lr=lr_now,
                            train=agg_tr, val=agg_vl))

        if agg_vl['total'] < best_val:
            best_val   = agg_vl['total']
            no_improve = 0
            torch.save(bundle.state_dict(), SAVE_ENC)
            torch.save(model.state_dict(),  SAVE_FULL)
            print(f"    [OK] Saved (val={best_val:.4f})")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"\n[*] Early stop at epoch {epoch+1}")
                break

    print(f"\n[*] Best val loss : {best_val:.4f}")
    print(f"[*] Encoders saved: {SAVE_ENC}")
    print(f"[*] Full saved    : {SAVE_FULL}")
    return SAVE_ENC, SAVE_FULL, history


# ── Run ──────────────────────────────────────────────────────────────────────
ENCODER_PATH, FULL_PATH, HISTORY = train_hybrid_v4(
    compiled_dirs = [MIMIC_COMPILED],
    epochs           = 100,
    batch_size       = 32,
    lr               = 1e-4,
    patience         = 20,
    M                = 3,
    tau              = 0.2,
    mask_ratio       = 0.25,# ← lowered from 0.40
    block_patches    = 6,
    lam_d1           = 0.3,
    lam_d2           = 0.5,
    lam_freq         = 0.5,
    lam_peak         = 0.3,
    lam_clean        = 1.0,             # NEW — fixes masked-region collapse
    lam_wavelet      = 0.01,            # NEW — anchors ECG wavelet filters
    lam_intra        = 1.0,
    
    lam_cross        = 0.1,
    peak_quantile    = 0.85,
    d_model          = 128,
    n_heads          = 8,
    private_layers   = 2,
    shared_layers    = 3,
    n_wavelet_levels = 3,
)

print("\n[OK] Hybrid v4 pretraining complete.")
print(f"     Encoders: {ENCODER_PATH}")
print(f"     Full:     {FULL_PATH}")


## CELL 7: RECONSTRUCTION VISUALIZATION

In [ ]:
# ==============================================================================
# CELL 7: RECONSTRUCTION VISUALIZATION  —  6 SEPARATE FIGURES
#
# NOTE: now plots the CLEAN-path reconstruction (out['x_clean_*']) by default,
# because that's the one with the direct MSE objective and the one that should
# in-paint masked regions well.  Set show_aggregate=True to also dump the
# SimMTM-aggregate reconstruction figures alongside (12 figures total).
# ==============================================================================
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

CHANNEL_COLORS = {'ppg': '#378ADD', 'abp': '#1D9E75', 'ecg': '#D85A30'}
FULL_PATH = "/kaggle/working/hybrid_v4_full.pth"
MIMIC_COMPILED = "/kaggle/working/mimic_memmap"
def robust_z(x, clip=5.0):
    med = np.median(x)
    q75, q25 = np.percentile(x, [75, 25])
    iqr = (q75 - q25) + 1e-6
    z = (x - med) / iqr
    z = np.clip(z, -clip, clip)
    return z.astype(np.float32)


def _make_one_figure(t_axis, gt, pred, ch, sample_i, kind, save_path):
    """One reconstruction figure for one (sample, modality, kind)."""
    mse = float(np.mean((gt - pred) ** 2))

    fig, ax = plt.subplots(1, 1, figsize=(18, 4.2), facecolor='#0f0f0f')
    ax.set_facecolor('#1a1a1a')
    for sp in ax.spines.values():
        sp.set_edgecolor('#444444')

    ax.plot(t_axis, gt,   color=CHANNEL_COLORS[ch],
            lw=1.2, alpha=0.9,  label='Ground truth (robust z)')
    ax.plot(t_axis, pred, color='#FFD700',
            lw=1.4, alpha=0.95, ls='--', label='Reconstruction')

    for p_idx in range(125):
        s = p_idx * 30
        e = s + 30
        patch_mse = float(np.mean((gt[s:e] - pred[s:e]) ** 2))
        

    ax.set_title(
        f'Hybrid v4 reconstruction ({kind})  |  Sample {sample_i}  |  {ch.upper()}  '
        f'|  MSE = {mse:.6f}',
        color='white', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlim(0, 3750)
    ymin = min(gt.min(), pred.min()) - 0.3
    ymax = max(gt.max(), pred.max()) + 0.3
    ax.set_ylim(ymin, ymax)
    ax.tick_params(colors='#888888', labelsize=8)
    ax.set_xlabel('Sample index',       color='#888888', fontsize=10)
    ax.set_ylabel('Robust-z amplitude', color='#888888', fontsize=10)

    handles = [
        plt.Line2D([0], [0], color=CHANNEL_COLORS[ch], lw=1.5,
                   label='Ground truth (robust z)'),
        plt.Line2D([0], [0], color='#FFD700', lw=1.5, ls='--',
                   label='Reconstruction'),
    
    ]
    ax.legend(handles=handles, loc='upper right',
              facecolor='#2a2a2a', edgecolor='#444444',
              labelcolor='white', fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches='tight', facecolor='#0f0f0f')
    plt.show()
    plt.close(fig)
    return mse


def visualize_recon_v4_separate(full_path, compiled_dir, device,
                                n_samples=2, seed=99,
                                d_model=128, n_heads=8,
                                private_layers=2, shared_layers=3,
                                n_wavelet_levels=3,
                                save_dir="/kaggle/working",
                                show_aggregate=False):
    """
    n_samples × 3 modalities figures (clean-path recon).
    If show_aggregate=True: also plots the SimMTM aggregate recon (doubles output).
    """
    if not os.path.exists(full_path):
        print(f"  [!] {full_path} not found — skipping.")
        return

    N = int(np.load(os.path.join(compiled_dir, 'n_samples.npy'))[0])
    memmaps = {ch: np.memmap(os.path.join(compiled_dir, f'{ch}.dat'),
                              dtype='float32', mode='r', shape=(N, 3750))
               for ch in ['ppg', 'abp', 'ecg']}

    enc_ppg = PrivateEncoderRaw(
        seq_len=3750, patch_len=30, d_model=d_model,
        n_heads=n_heads, n_layers=private_layers).to(device)
    enc_abp = PrivateEncoderRaw(
        seq_len=3750, patch_len=30, d_model=d_model,
        n_heads=n_heads, n_layers=private_layers).to(device)
    enc_ecg = PrivateEncoderECG(
        seq_len=3750, patch_len=30, d_model=d_model,
        n_heads=n_heads, n_layers=private_layers,
        n_wavelet_levels=n_wavelet_levels).to(device)
    shared  = SharedCrossModalEncoder(
        d_model=d_model, n_heads=n_heads, n_layers=shared_layers,
        num_patches=3750 // 30).to(device)

    model_viz = HybridPretrainerV4(
        enc_ppg=enc_ppg, enc_abp=enc_abp, enc_ecg=enc_ecg, shared=shared,
        d_model=d_model, patch_len=30, num_patches=3750 // 30,
        M=3, tau=0.2, mask_ratio=0.25, block_patches=6,
    ).to(device)
    model_viz.load_state_dict(torch.load(full_path, map_location=device))
    model_viz.eval()

    rng      = np.random.RandomState(seed)
    indices  = rng.randint(0, N, size=n_samples).tolist()
    t_axis   = np.arange(3750)
    channels = ['ppg', 'abp', 'ecg']
    saved    = []

    with torch.no_grad():
        for s_i, idx in enumerate(indices):
            x_ppg = torch.tensor(robust_z(np.array(memmaps['ppg'][idx]))
                                 ).unsqueeze(0).to(device)
            x_abp = torch.tensor(robust_z(np.array(memmaps['abp'][idx]))
                                 ).unsqueeze(0).to(device)
            x_ecg = torch.tensor(robust_z(np.array(memmaps['ecg'][idx]))
                                 ).unsqueeze(0).to(device)

            out = model_viz(x_ppg, x_abp, x_ecg)

            inputs       = {'ppg': x_ppg, 'abp': x_abp, 'ecg': x_ecg}
            recons_clean = {'ppg': out['x_clean_ppg'],
                            'abp': out['x_clean_abp'],
                            'ecg': out['x_clean_ecg']}
            recons_agg   = {'ppg': out['x_hat_ppg'],
                            'abp': out['x_hat_abp'],
                            'ecg': out['x_hat_ecg']}

            for ch in channels:
                # CLEAN-path figure (primary)
                gt   = inputs[ch][0].cpu().numpy()
                pred = recons_clean[ch][0].cpu().numpy()
                p    = os.path.join(save_dir, f"recon_v4_sample{s_i+1}_{ch}_clean.png")
                mse  = _make_one_figure(t_axis, gt, pred, ch, s_i+1, "clean", p)
                print(f"[OK] Saved {p}   (clean MSE={mse:.6f})")
                saved.append(p)

                # SimMTM-aggregate figure (optional)
                if show_aggregate:
                    pred_a = recons_agg[ch][0].cpu().numpy()
                    p_a    = os.path.join(save_dir,
                                          f"recon_v4_sample{s_i+1}_{ch}_aggregate.png")
                    mse_a  = _make_one_figure(t_axis, gt, pred_a, ch, s_i+1,
                                              "SimMTM agg", p_a)
                    print(f"[OK] Saved {p_a}  (agg MSE={mse_a:.6f})")
                    saved.append(p_a)

    print(f"\n[OK] Wrote {len(saved)} figures to {save_dir}")
    return saved


visualize_recon_v4_separate(FULL_PATH, MIMIC_COMPILED, DEVICE,
                            n_samples=2, seed=99,
                            show_aggregate=True)
